# Cross-Model Activation Transfer


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
from umap import UMAP

from activation_cross_analysis import (
    calc_bidirectional_knn,
    calc_cross_similarity,
    calc_transport_latents,
)
from capture.saving import load_captured_activations


# Setup

Load captured activations once and convert tensors to NumPy matrices with neurons as rows and examples as columns.


In [ ]:
results_path = Path("results")
captured = load_captured_activations(results_path)
tasks = tuple(captured)

activations = {}
for task in tasks:
    activations[task] = {
        "base": captured[task]["base"].states.detach().cpu().numpy().T,
        "finetuned": captured[task]["finetuned"].states.detach().cpu().numpy().T,
    }


# Plot Placeholders

These are intentionally empty until plotting is implemented.


In [ ]:
def sampled_positions(length, max_ticks=10):
    if length <= 0:
        return np.array([], dtype=int)
    tick_count = min(max_ticks, length)
    return np.unique(np.linspace(0, length - 1, tick_count, dtype=int))


def set_neuron_ticks(axis, positions, labels, orientation):
    if orientation == "x":
        axis.set_xticks(positions)
        axis.set_xticklabels(labels, rotation=90)
    else:
        axis.set_yticks(positions)
        axis.set_yticklabels(labels)


def signed_heatmap(axis, matrix, title, x_order=None, y_order=None):
    matrix = np.asarray(matrix, dtype=float)
    max_abs = np.nanmax(np.abs(matrix)) if matrix.size else 1.0
    color_limit = max_abs if np.isfinite(max_abs) and max_abs > 0.0 else 1.0
    image = axis.imshow(
        matrix,
        cmap="coolwarm",
        vmin=-color_limit,
        vmax=color_limit,
        interpolation="nearest",
        aspect="auto",
    )
    axis.set_title(title)
    axis.set_xlabel("fine-tuned neuron")
    axis.set_ylabel("base neuron")

    y_order = np.arange(matrix.shape[0]) if y_order is None else np.asarray(y_order)
    x_order = np.arange(matrix.shape[1]) if x_order is None else np.asarray(x_order)
    y_positions = sampled_positions(matrix.shape[0])
    x_positions = sampled_positions(matrix.shape[1])
    set_neuron_ticks(axis, x_positions, [str(int(x_order[position])) for position in x_positions], "x")
    set_neuron_ticks(axis, y_positions, [str(int(y_order[position])) for position in y_positions], "y")
    return image


def draw_spectral_ordering(task, metric, ordered_W, base_order, finetuned_order):
    figure, axis = plt.subplots(figsize=(10, 8))
    image = signed_heatmap(
        axis,
        ordered_W,
        f"{task} {metric} SVD spectral ordering",
        x_order=finetuned_order,
        y_order=base_order,
    )
    figure.colorbar(image, ax=axis, label=metric)
    figure.tight_layout()
    plt.show()


def draw_singular_values(task, metric, singular_values):
    ranks = np.arange(1, len(singular_values) + 1)
    figure, axis = plt.subplots(figsize=(8, 5))
    axis.plot(ranks, singular_values, marker="o", linewidth=1.5, markersize=3)
    axis.set_title(f"{task} {metric} singular values")
    axis.set_xlabel("rank")
    axis.set_ylabel("singular value")
    axis.grid(True, alpha=0.3)
    figure.tight_layout()
    plt.show()


def project_transport_latents(Pb, Pf):
    latents = np.vstack([Pb, Pf])
    neighbor_count = min(15, max(2, latents.shape[0] - 1))
    projected = UMAP(n_components=2, n_neighbors=neighbor_count, random_state=0).fit_transform(latents)

    return projected[: Pb.shape[0]], projected[Pb.shape[0] :]


def draw_transport_latents(task, metric, base_xy, finetuned_xy, rank, projection_label):
    base_xy = np.asarray(base_xy, dtype=float)
    finetuned_xy = np.asarray(finetuned_xy, dtype=float)

    figure, axis = plt.subplots(figsize=(10, 8))
    axis.scatter(base_xy[:, 0], base_xy[:, 1], color="tab:blue", label="base", s=18, alpha=0.75)
    axis.scatter(finetuned_xy[:, 0], finetuned_xy[:, 1], color="tab:orange", label="fine-tuned", s=18, alpha=0.75)

    for neuron_index, (x, y) in enumerate(base_xy):
        axis.text(x, y, str(neuron_index), fontsize=5, color="tab:blue", alpha=0.65)
    for neuron_index, (x, y) in enumerate(finetuned_xy):
        axis.text(x, y, str(neuron_index), fontsize=5, color="tab:orange", alpha=0.65)

    axis.set_title(f"{task} {metric} rank-{rank} transport latents via {projection_label}")
    axis.set_xlabel(f"{projection_label} dimension 1")
    axis.set_ylabel(f"{projection_label} dimension 2")
    axis.legend()
    axis.grid(True, alpha=0.25)
    figure.tight_layout()
    plt.show()


def draw_transport_knn_graph(
    task,
    metric,
    query_indices,
    candidate_indices,
    scores,
    candidate_degrees,
    rank,
    direction,
    hub_count=12,
    edges_per_hub=24,
):
    candidate_degrees = np.asarray(candidate_degrees)
    if candidate_degrees.size == 0:
        return

    hub_indices = np.argsort(-candidate_degrees, kind="stable")[:hub_count]
    selected_positions = []
    for hub_index in hub_indices:
        positions = np.flatnonzero(candidate_indices == hub_index)
        positions = positions[np.argsort(-scores[positions], kind="stable")]
        selected_positions.extend(positions[:edges_per_hub].tolist())
    selected_positions = np.unique(selected_positions)

    source_is_base = direction.startswith("base")
    source_prefix = "B" if source_is_base else "F"
    target_prefix = "F" if source_is_base else "B"
    network = nx.DiGraph()
    for position in selected_positions:
        source = f"{source_prefix}{int(query_indices[position])}"
        target = f"{target_prefix}{int(candidate_indices[position])}"
        network.add_edge(
            source,
            target,
            score=float(scores[position]),
        )

    if not network.nodes:
        return
    source_nodes = [node for node in network if node.startswith(source_prefix)]
    target_nodes = [node for node in network if node.startswith(target_prefix)]
    positions = nx.bipartite_layout(network, source_nodes, align="vertical")
    degrees = dict(network.degree())
    figure, axis = plt.subplots(figsize=(12, 9))
    nx.draw_networkx_nodes(
        network,
        positions,
        nodelist=source_nodes,
        node_color="tab:blue",
        node_size=[50 + 25 * degrees[node] for node in source_nodes],
        alpha=0.85,
        ax=axis,
    )
    nx.draw_networkx_nodes(
        network,
        positions,
        nodelist=target_nodes,
        node_color="tab:orange",
        node_size=[50 + 25 * degrees[node] for node in target_nodes],
        alpha=0.9,
        ax=axis,
    )
    edge_scores = np.array([data["score"] for _, _, data in network.edges(data=True)])
    score_range = np.ptp(edge_scores) if edge_scores.size else 0.0
    edge_widths = (
        0.75 + 2.5 * (edge_scores - edge_scores.min()) / score_range
        if score_range > 0.0
        else np.full(edge_scores.shape, 1.5)
    )
    nx.draw_networkx_edges(
        network,
        positions,
        width=edge_widths,
        alpha=0.35,
        arrows=True,
        arrowsize=8,
        arrowstyle="-|>",
        ax=axis,
    )
    hub_labels = {
        f"{target_prefix}{int(index)}": f"{target_prefix}{int(index)}"
        for index in hub_indices
        if f"{target_prefix}{int(index)}" in network
    }
    nx.draw_networkx_labels(network, positions, labels=hub_labels, font_size=8, ax=axis)
    axis.set_title(f"{task} {metric} rank-{rank} {direction} KNN hubs")
    axis.axis("off")
    figure.tight_layout()
    plt.show()


### Cross Similarity

Build the shared cross-model similarity source W for downstream cells.


In [ ]:
similarity_by_task_metric = {}
for task in tasks:
    base_states = activations[task]["base"]
    finetuned_states = activations[task]["finetuned"]
    pearson, cosine = calc_cross_similarity(base_states, finetuned_states)
    similarity_by_task_metric[task, "pearson"] = pearson
    similarity_by_task_metric[task, "cosine"] = cosine


### Shared SVD Artifacts

Compute the SVD of each W exactly once. Singular values, residuals, and low-rank transport cells reuse these artifacts.


In [ ]:
svd_by_task_metric = {}
for (task, metric), W in similarity_by_task_metric.items():
    svd_by_task_metric[task, metric] = np.linalg.svd(W, full_matrices=False)


# CKA

Compute one CKA score per task directly from X and Y, then draw it.


In [ ]:
for task in tasks:
    X = activations[task]["base"]
    Y = activations[task]["finetuned"]

    X_centered = X - X.mean(axis=1, keepdims=True)
    Y_centered = Y - Y.mean(axis=1, keepdims=True)

    numerator = np.linalg.norm(np.matmul(X_centered, Y_centered.T), ord="fro") ** 2
    denominator = np.linalg.norm(np.matmul(X_centered, X_centered.T), ord="fro") * np.linalg.norm(
        np.matmul(Y_centered, Y_centered.T),
        ord="fro",
    )
    cka_score = np.nan if denominator == 0.0 else numerator / denominator

    print(task,"cka:",cka_score)

# Spectral Ordering

Compute the spectral ordering mathematics from W and draw the reordered signed matrix. This SVD is on the normalized affinity used for ordering, not the reusable SVD of W.


In [ ]:
for (task, metric), W in similarity_by_task_metric.items():
    affinity = np.abs(W)
    row_degree = affinity.sum(axis=1)
    column_degree = affinity.sum(axis=0)
    normalized = np.divide(
        affinity,
        np.sqrt(row_degree[:, None] * column_degree[None, :]),
        out=np.zeros_like(affinity, dtype=float),
        where=(row_degree[:, None] > 0.0) & (column_degree[None, :] > 0.0),
    )

    spectral_U, spectral_s, spectral_Vt = np.linalg.svd(normalized, full_matrices=False)
    vector_index = 1 if spectral_s.size > 1 else 0
    base_order = np.argsort(spectral_U[:, vector_index], kind="mergesort")
    finetuned_order = np.argsort(spectral_Vt[vector_index], kind="mergesort")
    ordered_W = W[base_order][:, finetuned_order]

    draw_spectral_ordering(task, metric, ordered_W, base_order, finetuned_order)


# Singular Values

Draw singular values from the shared SVD artifacts.


In [ ]:
for (task, metric), (_, singular_values, _) in svd_by_task_metric.items():
    draw_singular_values(task, metric, singular_values)


# Bidirectional KNN Transport Graphs

Compute rank-64 base and fine-tuned latent points, then draw separate base-to-fine and fine-to-base bipartite KNN graphs.


In [ ]:
transport_rank = 64
knn_k = 5

for (task, metric), (U, singular_values, Vt) in svd_by_task_metric.items():
    k_used = min(transport_rank, singular_values.size)
    Pb, Pf = calc_transport_latents(U, singular_values, Vt, k_used)

    base_xy, finetuned_xy = project_transport_latents(Pb, Pf)
    draw_transport_latents(task, metric, base_xy, finetuned_xy, k_used, "UMAP")

    base_to_finetuned, finetuned_to_base = calc_bidirectional_knn(Pb, Pf, k=knn_k)
    base_queries, base_candidates, base_scores, base_degrees = base_to_finetuned
    finetuned_queries, finetuned_candidates, finetuned_scores, finetuned_degrees = finetuned_to_base
    draw_transport_knn_graph(
        task, metric, base_queries, base_candidates, base_scores, base_degrees, k_used, "base -> fine-tuned"
    )
    draw_transport_knn_graph(
        task, metric, finetuned_queries, finetuned_candidates, finetuned_scores, finetuned_degrees, k_used, "fine-tuned -> base"
    )
